In [ ]:
from pydantic import BaseModel, Field
from typing import Literal,TypedDict
from langgraph.graph import StateGraph, START, END
from langchain.messages import HumanMessage, SystemMessage

from langchain_deepseek import ChatDeepSeek

from dotenv import load_dotenv
load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)

# 構造化出力用の Schema を定義し、ルーティング判断の根拠とする
class Route(BaseModel):
    step: Literal["poem", "story", "joke"] = Field(
        None,
        description="ルーティングフローにおける次の実行ステップ",
    )

# 大規模言語モデルに構造化出力機能を追加
router = model.with_structured_output(Route)

# グラフの状態
class OverAllState(TypedDict):
    input: str
    decision: str
    output: str

# ノード
def model_call_1(state: OverAllState) -> OverAllState:
    """物語を生成する"""

    result = model.invoke(
        [HumanMessage(content=state["input"])]
    )
    return {"output": result.content}


def model_call_2(state: OverAllState) -> OverAllState:
    """ジョークを生成する"""

    result = model.invoke(
        [HumanMessage(content=state["input"])]
    )
    return {"output": result.content}


def model_call_3(state: OverAllState) -> OverAllState:
    """詩を生成する"""

    result = model.invoke(
        [HumanMessage(content=state["input"])]
    )
    return {"output": result.content}


def model_call_router(state: OverAllState) -> OverAllState:
    """ユーザーの入力を適切なノードにルーティングする"""

    # 構造化出力機能を持つ大規模言語モデルを呼び出し、ルーティング判断を行う
    decision = router.invoke(
        [
            SystemMessage(
                content=(
                    "ユーザーのリクエストに基づいて、story、joke、poem のいずれかにルーティングしてください。"
                    "物語の作成を求められた場合は story を、ジョークの作成を求められた場合は joke を、"
                    "詩の作成を求められた場合は poem を返してください。"
                )
            ),
            HumanMessage(content=state["input"]),
        ]
    )

    return {"decision": decision.step}


# 条件付きエッジ関数：ルーティング決定に基づいて次のノードを選択
def route_decision(
        state: OverAllState
) -> Literal[
    "model_call_1",
    "model_call_2",
    "model_call_3",
    END
]:
    # 次に実行するノード名を返す
    if state["decision"] == "story":
        return "model_call_1"
    elif state["decision"] == "joke":
        return "model_call_2"
    elif state["decision"] == "poem":
        return "model_call_3"
    return END


# ワークフローを構築
builder = StateGraph(OverAllState)

# ノードを追加
builder.add_node("model_call_1", model_call_1)
builder.add_node("model_call_2", model_call_2)
builder.add_node("model_call_3", model_call_3)
builder.add_node("model_call_router", model_call_router)

# エッジを追加して各ノードを接続
builder.add_edge(START, "model_call_router")
builder.add_conditional_edges(
    "model_call_router",
    route_decision,
    {
        # route_decision が返す名前：次に実行するノード名
        "model_call_1": "model_call_1",
        "model_call_2": "model_call_2",
        "model_call_3": "model_call_3",
    },
)
builder.add_edge("model_call_1", END)
builder.add_edge("model_call_2", END)
builder.add_edge("model_call_3", END)

# ワークフローをコンパイル
graph = builder.compile()

# ワークフローを呼び出す
state = graph.invoke({"input": "猫についての詩を書いてください"})
print(state["output"])

# ワークフローグラフを表示
from IPython.display import display
display(graph)